# QC check 2 — stage 1: regional neighbour statistics (DuckDB/Parquet)

Stage one of the second QC check. For every **located** station-day (a station
with an assigned `matched_latitude` / `matched_longitude` / `matched_year`) this
computes robust neighbour statistics drawn from station-days that **passed** the
first QC check (`daily_qc_status.final_flag = 'pass'`):

* median of the neighbours' consensus rainfall for the same calendar day,
* the number of such neighbours, and
* the median absolute deviation (MAD) of the neighbours' values,

at 20 km and 50 km. The consensus station itself is always excluded.

"Same calendar day" = same `matched_year` + `month` + `day_of_month`. Distances
use an equirectangular approximation with a lat/lon bounding-box pre-filter.

Output: a `regional_daily_stats` Parquet table under `regional_root`, one row
per located target station-day (targets with no neighbours are emitted with
counts 0 and NULL stats).

In [1]:
# Setup roots and imports.

import os
from pathlib import Path

import duckdb

from src.rainfall_rescue_sqlite.parquet_ingest import default_ensemble_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root
from src.rainfall_rescue_sqlite.parquet_qc_exact_monthly import default_qc_parquet_root
from src.rainfall_rescue_sqlite.parquet_regional_stats import (
    compute_regional_daily_stats_parquet,
    default_regional_stats_parquet_root,
)

# Cap DuckDB memory and give it a disk spill dir so the local run cannot OOM
# the workstation (holistic medians spill instead of crashing).
os.environ.setdefault("DUCKDB_MEMORY_LIMIT", "3GB")
os.environ.setdefault("DUCKDB_TEMP_DIR", "/var/tmp/duckdb_regional_stats")
Path(os.environ["DUCKDB_TEMP_DIR"]).mkdir(parents=True, exist_ok=True)

pdir = Path(os.environ["PDIR"])
ensemble_dataset_root = default_ensemble_parquet_root()
comparison_root = default_comparison_parquet_root()
qc_root = default_qc_parquet_root()
regional_root = default_regional_stats_parquet_root()

print(f"ensemble dataset: {ensemble_dataset_root}")
print(f"comparison root:  {comparison_root}")
print(f"QC root:          {qc_root}")
print(f"regional root:    {regional_root}")

ensemble dataset: /data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet
comparison root:  /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
QC root:          /data/scratch/philip.brohan/ADRQ/qc_parquet
regional root:    /data/scratch/philip.brohan/ADRQ/regional_stats_parquet


In [2]:
# Smoke test on a small target file-id slice (neighbour pool is the full pass set).

result = compute_regional_daily_stats_parquet(
    ensemble_dataset_root=ensemble_dataset_root,
    comparison_root=comparison_root,
    qc_root=qc_root,
    output_root=regional_root,
    start_file_id=1,
    end_file_id=200,
)

result

RegionalStatsResult(metadata_session_id=1, qc_session_id=2, target_rows_written=43152, targets_with_neighbours=42718, output_path=PosixPath('/data/scratch/philip.brohan/ADRQ/regional_stats_parquet/regional_daily_stats/session_meta000001_qc000002_1_200.parquet'))

In [3]:
# Inspect the output schema and a few rows with neighbours.

conn = duckdb.connect()
try:
    schema = conn.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{result.output_path}')"
    ).df()
    sample = conn.execute(
        f"""
        SELECT file_id, matched_year, month, day_of_month, consensus_value,
               n_20km, median_20km, mad_20km, n_50km, median_50km, mad_50km
        FROM read_parquet('{result.output_path}')
        WHERE n_20km > 0
        ORDER BY n_20km DESC
        LIMIT 10
        """
    ).df()
finally:
    conn.close()

print(schema)
sample

            column_name column_type null   key default extra
0               file_id      BIGINT  YES  None    None  None
1          matched_year      BIGINT  YES  None    None  None
2                 month     TINYINT  YES  None    None  None
3          day_of_month     TINYINT  YES  None    None  None
4       consensus_value      DOUBLE  YES  None    None  None
5                n_20km      BIGINT  YES  None    None  None
6           median_20km      DOUBLE  YES  None    None  None
7              mad_20km      DOUBLE  YES  None    None  None
8                n_50km      BIGINT  YES  None    None  None
9           median_50km      DOUBLE  YES  None    None  None
10             mad_50km      DOUBLE  YES  None    None  None
11  metadata_session_id      BIGINT  YES  None    None  None
12        qc_session_id      BIGINT  YES  None    None  None
13           created_at     VARCHAR  YES  None    None  None


,file_id,matched_year,month,day_of_month,consensus_value,n_20km,median_20km,mad_20km,n_50km,median_50km,mad_50km
0,178,1881,4,1,0.0,19,0.0,0.0,60,0.0,0.0
1,178,1881,4,2,0.0,19,0.0,0.0,60,0.0,0.0
2,178,1881,4,3,0.0,19,0.0,0.0,60,0.0,0.0
3,178,1881,4,4,0.0,19,0.0,0.0,60,0.0,0.0
4,178,1881,4,5,0.0,19,0.0,0.0,60,0.0,0.0
5,178,1881,4,6,0.0,19,0.0,0.0,60,0.0,0.0
6,178,1881,4,7,0.0,19,0.0,0.0,60,0.0,0.0
7,178,1881,4,8,0.0,19,0.0,0.0,60,0.0,0.0
8,178,1881,4,9,0.0,19,0.0,0.0,60,0.0,0.0
9,178,1881,4,10,0.0,19,0.0,0.0,60,0.0,0.0


In [7]:
# Sanity checks: counts monotone, non-negative stats, self excluded.

conn = duckdb.connect()
try:
    checks = conn.execute(
        f"""
        SELECT
            COUNT(*)                                                    AS rows,
            SUM(CASE WHEN n_20km > n_50km THEN 1 ELSE 0 END)            AS bad_counts,
            SUM(CASE WHEN mad_20km < 0 OR mad_50km < 0 THEN 1 ELSE 0 END) AS bad_mad,
            SUM(CASE WHEN median_20km < 0 OR median_50km < 0 THEN 1 ELSE 0 END) AS bad_median,
            SUM(CASE WHEN n_50km > 0 THEN 1 ELSE 0 END)                 AS with_neighbours,
            SUM(CASE WHEN n_20km > 0 AND median_20km IS NULL THEN 1 ELSE 0 END) AS missing_median20
        FROM read_parquet('{result.output_path}')
        """
    ).fetchone()
finally:
    conn.close()

rows, bad_counts, bad_mad, bad_median, with_neighbours, missing_median20 = checks
print(f"rows                : {rows}")
print(f"rows w/ neighbours  : {with_neighbours}")
print(f"n_20km > n_50km     : {bad_counts}")
print(f"negative MAD        : {bad_mad}")
print(f"negative median     : {bad_median}")
print(f"n_20km>0 & null med : {missing_median20}")

assert bad_counts == 0, "n_20km should never exceed n_50km"
assert bad_mad == 0, "MAD must be non-negative"
assert bad_median == 0, "median rainfall must be non-negative"
assert missing_median20 == 0, "median_20km must be set whenever n_20km>0"
print("\nAll sanity checks passed.")

rows                : 43152
rows w/ neighbours  : 42718
n_20km > n_50km     : 0
negative MAD        : 0
negative median     : 0
n_20km>0 & null med : 0

All sanity checks passed.


## Independent hand-verification

Pick one target station-day with several neighbours, fetch its neighbour
`(value, distance)` rows directly, and recompute count / median / MAD in Python
with `statistics` — fully independent of the module's SQL — then compare.

In [5]:
import math
import statistics

EARTH_RADIUS_KM = 6371.0
metadata_glob = f"{comparison_root / 'ensemble_metadata' / '*.parquet'}"
daily_glob = f"{ensemble_dataset_root / 'ensemble_daily_values' / '*.parquet'}"
status_glob = f"{qc_root / 'daily_qc_status' / '*.parquet'}"

conn = duckdb.connect()
try:
    # Pick a target station-day that actually exercises the MAD (spread > 0).
    tgt = conn.execute(
        f"""
        SELECT file_id, matched_year, month, day_of_month, consensus_value,
               n_20km, median_20km, mad_20km, n_50km, median_50km, mad_50km,
               metadata_session_id, qc_session_id
        FROM read_parquet('{result.output_path}')
        WHERE mad_20km > 0 AND n_20km >= 5
        ORDER BY n_20km DESC
        LIMIT 1
        """
    ).fetchdf().iloc[0]

    fid = int(tgt.file_id)
    yr, mo, dy = int(tgt.matched_year), int(tgt.month), int(tgt.day_of_month)
    msid, qsid = int(tgt.metadata_session_id), int(tgt.qc_session_id)

    # Target location.
    tlat, tlon = conn.execute(
        f"""
        SELECT matched_latitude, matched_longitude
        FROM read_parquet('{metadata_glob}')
        WHERE match_source_session_id = {msid} AND file_id = {fid}
        """
    ).fetchone()

    # Independently gather every candidate neighbour's location + consensus value:
    # a located station of the SAME matched_year that passed QC1 on this month/day.
    nb = conn.execute(
        f"""
        WITH meta AS (
            SELECT file_id, matched_latitude AS lat, matched_longitude AS lon
            FROM read_parquet('{metadata_glob}')
            WHERE match_source_session_id = {msid}
              AND matched_year = {yr}
              AND matched_latitude IS NOT NULL
              AND matched_longitude IS NOT NULL
        ),
        pass_day AS (
            SELECT file_id FROM read_parquet('{status_glob}')
            WHERE qc_session_id = {qsid} AND final_flag = 'pass'
              AND month = {mo} AND day_of_month = {dy}
        ),
        val AS (
            SELECT file_id, median(COALESCE(rainfall, 0.0)) AS value
            FROM read_parquet('{daily_glob}')
            WHERE month = {mo} AND day_of_month = {dy}
            GROUP BY file_id
        )
        SELECT m.file_id, m.lat, m.lon, v.value
        FROM meta m
        JOIN pass_day p USING (file_id)
        JOIN val v USING (file_id)
        WHERE m.file_id <> {fid}
        """
    ).fetchall()
finally:
    conn.close()


def equirect_km(lat1, lon1, lat2, lon2):
    x = math.radians(lon2 - lon1) * math.cos(math.radians((lat1 + lat2) / 2.0))
    y = math.radians(lat2 - lat1)
    return EARTH_RADIUS_KM * math.sqrt(x * x + y * y)


def stats(vals):
    if not vals:
        return 0, None, None
    med = statistics.median(vals)
    mad = statistics.median([abs(v - med) for v in vals])
    return len(vals), med, mad


dists = [(equirect_km(tlat, tlon, la, lo), v) for _, la, lo, v in nb]
v20 = [v for d, v in dists if d <= 20.0]
v50 = [v for d, v in dists if d <= 50.0]
n20, med20, mad20 = stats(v20)
n50, med50, mad50 = stats(v50)

print(f"target file_id={fid} year={yr} {yr}-{mo:02d}-{dy:02d}  ({tlat:.4f}, {tlon:.4f})")
print(f"                stored      independent")
print(f"n_20km        : {int(tgt.n_20km):>8}   {n20:>8}")
print(f"median_20km   : {tgt.median_20km:>8.4f}   {med20:>8.4f}")
print(f"mad_20km      : {tgt.mad_20km:>8.4f}   {mad20:>8.4f}")
print(f"n_50km        : {int(tgt.n_50km):>8}   {n50:>8}")
print(f"median_50km   : {tgt.median_50km:>8.4f}   {med50:>8.4f}")
print(f"mad_50km      : {tgt.mad_50km:>8.4f}   {mad50:>8.4f}")

assert n20 == int(tgt.n_20km)
assert n50 == int(tgt.n_50km)
assert math.isclose(med20, tgt.median_20km, abs_tol=1e-9)
assert math.isclose(med50, tgt.median_50km, abs_tol=1e-9)
assert math.isclose(mad20, tgt.mad_20km, abs_tol=1e-9)
assert math.isclose(mad50, tgt.mad_50km, abs_tol=1e-9)
print("\nIndependent recomputation matches the stored row.")


target file_id=178 year=1881 1881-04-11  (51.3603, -0.0308)
                stored      independent
n_20km        :       19         19
median_20km   :   0.1000     0.1000
mad_20km      :   0.0200     0.0200
n_50km        :       60         60
median_50km   :   0.1050     0.1050
mad_50km      :   0.0600     0.0600

Independent recomputation matches the stored row.


## Running at scale on SLURM

The smoke test above computes regional stats for a small slice of target
file IDs.  For the full dataset (~584,000 file IDs) run the distributed
pipeline from the cluster login node.

**Prerequisites (in order):**

1. **QC check 1** must have completed — the neighbour pool is drawn from
   station-days that passed that check (`daily_qc_status`).
2. **Daily consensus table** must be built.  Each day's consensus value is
   the ensemble median over ~1.1 billion member-day rows.  Computing that
   median *inside* the regional shards is fatal: a file-ID shard's neighbour
   pool is geographically scattered (file_id is not spatial), so it needs the
   consensus for a near-national set of stations, and DuckDB's grouped
   `median()` buffers every value and cannot spill — the array tasks get
   OOM-killed.  So the median is precomputed **once**, sharded by contiguous
   file_id (row-group pruning keeps each shard's memory bounded), and merged
   into a `daily_consensus` table.

Submit the two pipelines in order (each is an array + a merge that runs once
after every shard succeeds):

```
# 1. Build the daily-consensus table (must finish first)
scripts/slurm/submit_daily_consensus.sh

# 2. Compute regional stats (reads the consensus table — no in-shard median)
scripts/slurm/submit_regional_stats.sh
```

`submit_regional_stats.sh` guards on the consensus table existing and exits
with instructions if it is missing.  The cell below prints the current max
`file_id` so `CONSENSUS_TOTAL_FILE_IDS` / `REGIONAL_TOTAL_FILE_IDS` are set
correctly, along with the submit commands.


In [10]:
# Get the current max file_id so the *_TOTAL_FILE_IDS vars are set correctly.

conn = duckdb.connect()
try:
    max_file_id = conn.execute(
        f"SELECT MAX(file_id) FROM read_parquet('{ensemble_dataset_root / 'ensemble_files' / '*.parquet'}')"
    ).fetchone()[0]
finally:
    conn.close()

print(f"Max file_id: {max_file_id}")
print()
print("1. Build the daily-consensus table first:")
print(f"   CONSENSUS_TOTAL_FILE_IDS={max_file_id} scripts/slurm/submit_daily_consensus.sh")
print()
print("2. Then compute regional stats (reads the consensus table):")
print(f"   REGIONAL_TOTAL_FILE_IDS={max_file_id} scripts/slurm/submit_regional_stats.sh")
print()
print("Optionally use more regional shards for tighter neighbour pools:")
print(f"   REGIONAL_NUM_SHARDS=400 REGIONAL_TOTAL_FILE_IDS={max_file_id} scripts/slurm/submit_regional_stats.sh")


Max file_id: 584513

1. Build the daily-consensus table first:
   CONSENSUS_TOTAL_FILE_IDS=584513 scripts/slurm/submit_daily_consensus.sh

2. Then compute regional stats (reads the consensus table):
   REGIONAL_TOTAL_FILE_IDS=584513 scripts/slurm/submit_regional_stats.sh

Optionally use more regional shards for tighter neighbour pools:
   REGIONAL_NUM_SHARDS=400 REGIONAL_TOTAL_FILE_IDS=584513 scripts/slurm/submit_regional_stats.sh


## Inspect the full merged dataset

The cells above all read `result.output_path`, which is the **smoke-test
slice** (target file IDs 1–200, ~43k rows) written by the local run — not the
SLURM output.  The distributed merge writes the full dataset to a separate file
`session_meta<NNNNNN>_qc<NNNNNN>.parquet` (no `_start_end` suffix) in the same
`regional_daily_stats` directory.  The cell below loads that merged file and
re-runs the sanity checks against the whole dataset.
</VSCode.Cell>


In [11]:
# Load the full merged SLURM output (not the smoke-test slice) and check it.
import re

merged_dir = regional_root / "regional_daily_stats"
merged_files = [
    p
    for p in sorted(merged_dir.glob("session_meta*_qc*.parquet"))
    if re.fullmatch(r"session_meta\d+_qc\d+\.parquet", p.name)  # exclude _start_end slices
]
print("Merged datasets found:")
for p in merged_files:
    print(f"  {p.name}")
assert merged_files, "No merged dataset found — has the SLURM merge job completed?"

merged_path = merged_files[-1]

conn = duckdb.connect()
try:
    checks = conn.execute(
        f"""
        SELECT
            COUNT(*)                                                           AS rows,
            SUM(CASE WHEN n_50km > 0 THEN 1 ELSE 0 END)                        AS with_neighbours,
            SUM(CASE WHEN n_20km > n_50km THEN 1 ELSE 0 END)                   AS bad_counts,
            SUM(CASE WHEN mad_20km < 0 OR mad_50km < 0 THEN 1 ELSE 0 END)      AS bad_mad,
            SUM(CASE WHEN median_20km < 0 OR median_50km < 0 THEN 1 ELSE 0 END) AS bad_median,
            SUM(CASE WHEN n_20km > 0 AND median_20km IS NULL THEN 1 ELSE 0 END) AS missing_median20
        FROM read_parquet('{merged_path}')
        """
    ).fetchone()
finally:
    conn.close()

m_rows, m_with, m_bad_counts, m_bad_mad, m_bad_median, m_missing = checks
print(f"\nFull merged dataset : {merged_path.name}")
print(f"rows                : {m_rows}")
print(f"rows w/ neighbours  : {m_with}")
print(f"n_20km > n_50km     : {m_bad_counts}")
print(f"negative MAD        : {m_bad_mad}")
print(f"negative median     : {m_bad_median}")
print(f"n_20km>0 & null med : {m_missing}")

assert m_bad_counts == 0, "n_20km should never exceed n_50km"
assert m_bad_mad == 0, "MAD must be non-negative"
assert m_bad_median == 0, "median rainfall must be non-negative"
assert m_missing == 0, "median_20km must be set whenever n_20km>0"
print("\nAll sanity checks passed on the full merged dataset.")


Merged datasets found:
  session_meta000001_qc000002.parquet

Full merged dataset : session_meta000001_qc000002.parquet
rows                : 103565172
rows w/ neighbours  : 102355304
n_20km > n_50km     : 0
negative MAD        : 0
negative median     : 0
n_20km>0 & null med : 0

All sanity checks passed on the full merged dataset.


## Interactive regional-stat map

`scripts/diagnostics/plot_regional_stat_interactive.py` draws any one of the
regional statistics for a chosen day as an interactive
[Plotly](https://plotly.com/python/) map. Pick a **date** and a **stat name**
(`consensus_value`, `n_20km`, `median_20km`, `mad_20km`, `n_50km`,
`median_50km`, `mad_50km`) and every located station-day for that date is
plotted, coloured by the chosen statistic.

Hovering over a point shows the **specifier** (the ensemble transcription's
source file name), the station name, and the statistic's value; **clicking a
station copies its specifier to the clipboard**. Stations with no value for the
statistic (e.g. no neighbours) are drawn as pale grey circles.

The cell reads the full merged SLURM dataset if it is present, otherwise the
local smoke-test slice, and also writes a self-contained HTML file that opens in
a browser without a running kernel.


In [19]:
# Demonstrate the interactive regional-stat map, rendered inline.

import importlib.util
from datetime import date

from IPython.display import HTML, display

import src.rainfall_rescue_sqlite as _pkg

repo_root = Path(_pkg.__file__).resolve().parents[2]

regional_stat_map_script_path = (
    repo_root / "scripts" / "diagnostics" / "plot_regional_stat_interactive.py"
)
spec_regional_stat_map = importlib.util.spec_from_file_location(
    "regional_stat_interactive_plot", regional_stat_map_script_path
)
regional_stat_map_plot = importlib.util.module_from_spec(spec_regional_stat_map)
spec_regional_stat_map.loader.exec_module(regional_stat_map_plot)

# Choose a day and a stat to plot. Stat names are the numeric columns of
# regional_daily_stats: consensus_value, n_20km, median_20km, mad_20km,
# n_50km, median_50km, mad_50km.
target_date = date(1891, 11, 13)  # Change to any YYYY-MM-DD with matched records.
stat_name = "mad_50km"

regional_stat_map_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/"
    f"regional_{stat_name}_{target_date.isoformat()}.html"
)

# The script reads the merged full-dataset file under regional_root if present,
# otherwise every parquet shard there (which includes the smoke-test slice).
regional_stat_fig = regional_stat_map_plot.build_figure(
    stat=stat_name,
    target_date=target_date,
    comparison_root=comparison_root,
    regional_root=regional_root,
    output_path=regional_stat_map_output_path,
)

# Render as HTML (not fig.show()) so the click-to-copy JavaScript runs inline.
display(HTML(regional_stat_map_plot.inline_html(regional_stat_fig)))
